# CookLens vision training

Run All trains and evaluates the #67 EfficientNet-B2 candidate with the original 30 dishes, `other_food`, and `non_food`. It uses the pinned CookLens checkpoint and saves a new checkpoint and held-out report for each run.

The local configuration reads `ai/cv/runs/issue-67/inputs.json` when present. On a different machine, set the paths in the first cell to your data. See the [training notes](../../../docs/plan/2026-11-27-issue-67-vision-pilot.md) for the required files.

In [ ]:
from datetime import datetime, timezone
import json
import os
from pathlib import Path
import subprocess
import sys

cwd = Path.cwd()
repo = next((p for p in (cwd, *cwd.parents)
             if (p / "ai/cv/training/train_pilot.py").is_file()), None)
if repo is None:
    raise FileNotFoundError("Start the notebook kernel inside the CookLens checkout.")

run_home = repo / "ai/cv/runs/issue-67"
local_config = run_home / "inputs.json"
inputs = json.loads(local_config.read_text(encoding="utf-8")) if local_config.is_file() else {}
archives = Path(inputs.get("archives", repo / "ai/cv/data/30vnfoods")).expanduser().resolve()
fallback = Path(inputs.get("fallback", repo / "ai/cv/data/fallback-pilot")).expanduser().resolve()
prepared = Path(inputs.get("prepared", run_home / "prepared.json")).expanduser().resolve()
checkpoint = Path(inputs.get("checkpoint", repo / "deploy/artifacts/staging/vision/efficientnet_b2_best.pt")).expanduser().resolve()
output_root = Path(inputs.get("output_root", run_home)).expanduser().resolve()

epochs = 3
batch_size = 16
threads = 4
run_id = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S-%f")
output = output_root / run_id

required = [archives / f"{split}.zip" for split in ("train", "validation", "test")]
required += [fallback / "manifest.jsonl", fallback / "promotions.json", checkpoint]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError("Missing training inputs:\n" + "\n".join(missing))

def run_module(name, *args):
    command = [sys.executable, "-u", "-m", f"ai.cv.training.{name}", *map(str, args)]
    env = {**os.environ, "PYTHONUTF8": "1"}
    with subprocess.Popen(command, cwd=repo, env=env, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, encoding="utf-8", errors="replace") as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, command)

In [ ]:
promotions_file = fallback / "promotions.json"
promotions = json.loads(promotions_file.read_text(encoding="utf-8"))
dataset = json.loads(prepared.read_text(encoding="utf-8")) if prepared.is_file() else None
inputs_changed = dataset is not None and (
    dataset.get("promotions", {}) != promotions
    or (fallback / "manifest.jsonl").stat().st_mtime_ns > prepared.stat().st_mtime_ns
    or any(path.stat().st_mtime_ns > prepared.stat().st_mtime_ns for path in required[:3])
)
if dataset is None or inputs_changed:
    draft = prepared.with_name(f"{prepared.stem}-{run_id}.tmp")
    run_module("pilot_data", "--archives", archives, "--fallback", fallback, "--output", draft)
    draft.replace(prepared)
    dataset = json.loads(prepared.read_text(encoding="utf-8"))

print(f"Prepared {len(dataset['images']):,} photos for {len(dataset['classes'])} labels.")
for split, counts in dataset["fallback_counts"].items():
    print(split, counts)

In [ ]:
run_module("train_pilot", "--dataset", prepared, "--archives", archives, "--fallback", fallback,
           "--checkpoint", checkpoint, "--output", output, "--epochs", epochs,
           "--batch-size", batch_size, "--threads", threads)

In [ ]:
report_file = output / "report.json"
weights_file = output / "efficientnet_b2_pilot.pt"
report = json.loads(report_file.read_text(encoding="utf-8"))
if report["smoke"] or not weights_file.is_file():
    raise RuntimeError("The full training run did not produce its expected artifacts.")

print("Device", report["device"])
print("Selected epoch", report["best_epoch"])
print("Held-out top-3", f"{report['test']['top3']:.2%}")
print("Supported-dish top-3", f"{report['test']['supported_top3']:.2%}")
print("Held-out macro F1", f"{report['test']['macro_f1']:.4f}")
print("Checkpoint", weights_file)
print("Report", report_file)